<div style="text-align:right"><i>notebook by Claude Opus 4.8<br>June 2026</i></div>

# Project Euler 1–100: Claude Opus 4.8 Edition

This notebook contains solutions to Project Euler problems 1–100 written by **Claude Opus 4.8**. 

It uses the [`euler_data.py`](euler_data.py) framework from [**Peter Norvig's notebook**](Euler.ipynb).

## Setup

In [1]:
from euler_data import DATA, run, runs

In [106]:
"""Utility functions for Project Euler problems 1-100.

Covers the recurring themes in the first 100 problems: primes & factorization,
divisors & multiplicative functions, digits & palindromes & pandigitals,
combinatorics, figurate numbers, fractions/continued fractions, and a few
miscellaneous helpers.

Standard library is preferred for the classics so the functions are
self-contained and fast; sympy and numpy are used where they clearly help.
"""


from statistics import mean, median


import sys


import time


from bisect import bisect_left, bisect_right


from collections import Counter, defaultdict, deque


from fractions import Fraction


from functools import lru_cache, reduce


from heapq import heappush, heappop


from itertools import (
    accumulate,
    combinations,
    combinations_with_replacement,
    count,
    permutations,
    product,
    takewhile,
)


from math import (
    comb,
    factorial,
    floor,
    gcd,
    isqrt,
    lcm,
    perm,
    prod,
    sqrt,
)


import numpy as np


import sympy


sys.setrecursionlimit(1_000_000)


def primes_up_to(n):
    """Return a list of all primes <= n using a sieve of Eratosthenes."""
    if n < 2:
        return []
    sieve = bytearray([1]) * (n + 1)
    sieve[0] = sieve[1] = 0
    for i in range(2, isqrt(n) + 1):
        if sieve[i]:
            sieve[i * i::i] = bytearray(len(sieve[i * i::i]))
    return [i for i in range(n + 1) if sieve[i]]


def prime_sieve(n):
    """Boolean sieve: index i is True iff i is prime, for 0 <= i <= n."""
    if n < 2:
        return bytearray(n + 1)
    sieve = bytearray([1]) * (n + 1)
    sieve[0] = sieve[1] = 0
    for i in range(2, isqrt(n) + 1):
        if sieve[i]:
            sieve[i * i::i] = bytearray(len(sieve[i * i::i]))
    return sieve


def is_prime(n):
    """Deterministic primality test (Miller-Rabin) good for all 64-bit n."""
    if n < 2:
        return False
    for p in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):
        if n % p == 0:
            return n == p
    d = n - 1
    r = 0
    while d % 2 == 0:
        d //= 2
        r += 1
    for a in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):
        x = pow(a, d, n)
        if x == 1 or x == n - 1:
            continue
        for _ in range(r - 1):
            x = x * x % n
            if x == n - 1:
                break
        else:
            return False
    return True


def nth_prime(n):
    """Return the n-th prime (1-indexed): nth_prime(1) == 2."""
    return sympy.prime(n)


def primes_in_range(lo, hi):
    """Primes p with lo <= p <= hi (inclusive)."""
    return list(sympy.primerange(lo, hi + 1))


def factorize(n):
    """Return prime factorization as a dict {prime: exponent}."""
    factors = {}
    while n % 2 == 0:
        factors[2] = factors.get(2, 0) + 1
        n //= 2
    p = 3
    while p * p <= n:
        while n % p == 0:
            factors[p] = factors.get(p, 0) + 1
            n //= p
        p += 2
    if n > 1:
        factors[n] = factors.get(n, 0) + 1
    return factors


def divisors(n):
    """Return a sorted list of all positive divisors of n."""
    small, large = [], []
    for i in range(1, isqrt(n) + 1):
        if n % i == 0:
            small.append(i)
            if i != n // i:
                large.append(n // i)
    return small + large[::-1]


def num_divisors(n):
    """Number of positive divisors, d(n)."""
    result = 1
    for exp in factorize(n).values():
        result *= exp + 1
    return result


def sum_divisors(n):
    """Sum of all positive divisors, sigma(n) (includes n itself)."""
    result = 1
    for p, exp in factorize(n).items():
        result *= (p ** (exp + 1) - 1) // (p - 1)
    return result


def proper_divisor_sum(n):
    """Sum of proper divisors of n (divisors excluding n itself)."""
    if n <= 1:
        return 0
    return sum_divisors(n) - n


def divisor_sum_sieve(n):
    """Array s where s[i] == sum of proper divisors of i, for i <= n.

    Useful for amicable numbers, abundant/deficient (problems 21, 23).
    """
    s = [0] * (n + 1)
    for d in range(1, n // 2 + 1):
        for multiple in range(2 * d, n + 1, d):
            s[multiple] += d
    return s


def totient(n):
    """Euler's totient phi(n)."""
    result = n
    for p in factorize(n):
        result -= result // p
    return result


def totient_sieve(n):
    """Array phi where phi[i] == totient(i) for 0 <= i <= n."""
    phi = list(range(n + 1))
    for i in range(2, n + 1):
        if phi[i] == i:  # i is prime
            for j in range(i, n + 1, i):
                phi[j] -= phi[j] // i
    return phi


def digits(n, base=10):
    """Return list of digits of n, most-significant first."""
    if n == 0:
        return [0]
    n = abs(n)
    out = []
    while n:
        out.append(n % base)
        n //= base
    return out[::-1]


def from_digits(ds, base=10):
    """Inverse of digits(): build an integer from a list of digits."""
    return reduce(lambda acc, d: acc * base + d, ds, 0)


def digit_sum(n, base=10):
    """Sum of the digits of n."""
    return sum(digits(n, base))


def is_palindrome(n, base=10):
    """True if n reads the same forwards and backwards in the given base."""
    ds = digits(n, base)
    return ds == ds[::-1]


def is_pandigital(n, digit_set=None):
    """True if the digits of n are exactly digit_set (default '1'..'9').

    With no digit_set, checks for a 1-9 pandigital (each of 1-9 once).
    """
    s = str(n)
    if digit_set is None:
        digit_set = set("123456789")
    return len(s) == len(digit_set) and set(s) == digit_set


def is_permutation(a, b):
    """True if a and b are digit-permutations of each other."""
    return sorted(str(a)) == sorted(str(b))


def triangular(n):
    return n * (n + 1) // 2


def pentagonal(n):
    return n * (3 * n - 1) // 2


def hexagonal(n):
    return n * (2 * n - 1)


def is_triangular(t):
    n = (isqrt(8 * t + 1) - 1) // 2
    return n * (n + 1) // 2 == t


def is_pentagonal(p):
    """True if p is a (generalized-positive) pentagonal number."""
    n = (1 + isqrt(1 + 24 * p))
    return n % 6 == 0 and pentagonal(n // 6) == p


def is_hexagonal(h):
    n = (1 + isqrt(1 + 8 * h))
    return n % 4 == 0 and hexagonal(n // 4) == h


def fib():
    """Infinite generator of Fibonacci numbers: 1, 2, 3, 5, 8, ... (1-indexed F1=1, F2=2 style).

    Actually yields F1=1, F2=1, F3=2, ... the standard sequence.
    """
    a, b = 1, 1
    while True:
        yield a
        a, b = b, a + b


def fib_n(n):
    """Return the n-th Fibonacci number (F1 = F2 = 1) via fast doubling."""
    if n == 0:
        return 0
    def _fd(k):
        if k == 0:
            return (0, 1)
        a, b = _fd(k >> 1)
        c = a * (2 * b - a)
        d = a * a + b * b
        if k & 1:
            return (d, c + d)
        return (c, d)
    return _fd(n)[0]


def continued_fraction_sqrt(n):
    """Return (a0, [period]) of the continued fraction expansion of sqrt(n).

    Returns (a0, []) when n is a perfect square. Useful for problems 64-66.
    """
    a0 = isqrt(n)
    if a0 * a0 == n:
        return a0, []
    period = []
    m, d, a = 0, 1, a0
    while a != 2 * a0:
        m = d * a - m
        d = (n - m * m) // d
        a = (a0 + m) // d
        period.append(a)
    return a0, period


def convergents(a0, period, count_terms):
    """Yield (numerator, denominator) convergents of a continued fraction.

    period is repeated cyclically. Useful for Pell equations (problem 66)
    and sqrt(2) convergents (problem 57).
    """
    h_prev, h = 1, a0
    k_prev, k = 0, 1
    yield h, k
    i = 0
    for _ in range(count_terms - 1):
        a = period[i % len(period)] if period else 0
        i += 1
        h_prev, h = h, a * h + h_prev
        k_prev, k = k, a * k + k_prev
        yield h, k


def modpow(base, exp, mod):
    """Modular exponentiation (thin wrapper over built-in pow)."""
    return pow(base, exp, mod)


_ONES = ["", "one", "two", "three", "four", "five", "six", "seven", "eight",
         "nine", "ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen",
         "sixteen", "seventeen", "eighteen", "nineteen"]


_TENS = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy",
         "eighty", "ninety"]


def number_to_words(n):
    """Spell out an integer 0..999999 in (British-style) English words."""
    if n == 0:
        return "zero"

    def under_1000(x):
        words = []
        if x >= 100:
            words.append(_ONES[x // 100] + " hundred")
            x %= 100
            if x:
                words.append("and")
        if x >= 20:
            words.append(_TENS[x // 10])
            x %= 10
            if x:
                words.append(_ONES[x])
        elif x:
            words.append(_ONES[x])
        return " ".join(words)

    parts = []
    if n >= 1000:
        parts.append(under_1000(n // 1000) + " thousand")
        n %= 1000
    if n:
        parts.append(under_1000(n))
    return " ".join(parts)


def chunked(iterable, size):
    """Yield successive lists of length `size` from iterable."""
    it = iter(iterable)
    while True:
        block = []
        for _ in range(size):
            try:
                block.append(next(it))
            except StopIteration:
                if block:
                    yield block
                return
        yield block


def read_grid(text, sep=None, cast=int):
    """Parse a whitespace/line-delimited grid of numbers into a 2-D list.

    Handy for the bundled-data problems (e.g. 11, 18, 67, 81-83).
    """
    return [[cast(x) for x in line.split(sep)] for line in text.strip().splitlines()]

In [3]:
import operator


def _max_triangle_path(rows):
    """Maximum top-to-bottom path sum through a triangle (list of rows)."""
    dp = rows[-1][:]
    for r in range(len(rows) - 2, -1, -1):
        for c in range(len(rows[r])):
            dp[c] = rows[r][c] + max(dp[c], dp[c + 1])
    return dp[0]


def _read_matrix(text):
    return [[int(x) for x in line.split(",")]
            for line in text.splitlines() if line.strip()]


## [Problem 1](https://projecteuler.net/problem=1)

*Sum the integers below 1000 that are divisible by 3 or 5 (direct filter).*

In [4]:
def euler_1():
    """Multiples of 3 or 5"""
    return sum(n for n in range(1000) if n % 3 == 0 or n % 5 == 0)


run(euler_1)

  1: Multiples of 3 or 5                           0 msec ⇒ 233168            ✅ 

## [Problem 2](https://projecteuler.net/problem=2)

*Iterate the Fibonacci sequence below 4,000,000 and sum the even terms.*

In [5]:
def euler_2():
    """Even Fibonacci Numbers"""
    total = 0
    for f in fib():
        if f >= 4_000_000:
            break
        if f % 2 == 0:
            total += f
    return total


run(euler_2)

  2: Even Fibonacci Numbers                        0 msec ⇒ 4613732           ✅ 

## [Problem 3](https://projecteuler.net/problem=3)

*Factorize 600851475143 by trial division and take the largest prime factor.*

In [6]:
def euler_3():
    """Largest Prime Factor"""
    return max(factorize(600851475143))


run(euler_3)

  3: Largest Prime Factor                          0 msec ⇒ 6857              ✅ 

## [Problem 4](https://projecteuler.net/problem=4)

*Brute-force all 3-digit × 3-digit products, keeping the largest palindrome.*

In [7]:
def euler_4():
    """Largest Palindrome Product"""
    best = 0
    for a in range(100, 1000):
        for b in range(a, 1000):
            p = a * b
            if p > best and is_palindrome(p):
                best = p
    return best


run(euler_4)

  4: Largest Palindrome Product                   25 msec ⇒ 906609            ✅ 

## [Problem 5](https://projecteuler.net/problem=5)

*Fold the whole range 1..20 under lcm.*

In [8]:
def euler_5():
    """Smallest Multiple"""
    return reduce(lcm, range(1, 21))


run(euler_5)

  5: Smallest Multiple                             0 msec ⇒ 232792560         ✅ 

## [Problem 6](https://projecteuler.net/problem=6)

*Closed-form square-of-sum minus sum-of-squares for the first 100 naturals.*

In [9]:
def euler_6():
    """Sum Square Difference"""
    n = 100
    return (n * (n + 1) // 2) ** 2 - sum(i * i for i in range(1, n + 1))


run(euler_6)

  6: Sum Square Difference                         0 msec ⇒ 25164150          ✅ 

## [Problem 7](https://projecteuler.net/problem=7)

*Ask SymPy for the 10001st prime directly.*

In [10]:
def euler_7():
    """10001st Prime"""
    return nth_prime(10001)


run(euler_7)

  7: 10001st Prime                                 6 msec ⇒ 104743            ✅ 

## [Problem 8](https://projecteuler.net/problem=8)

*Slide a 13-digit window across the number and maximize the digit product.*

In [11]:
def euler_8():
    """Largest Product in a Series"""
    ds = [int(c) for line in DATA[8].split() for c in line]
    return max(prod(ds[i:i + 13]) for i in range(len(ds) - 12))


run(euler_8)

  8: Largest Product in a Series                   0 msec ⇒ 23514624000       ✅ 

## [Problem 9](https://projecteuler.net/problem=9)

*Brute-force a and b with c = 1000 − a − b, testing a² + b² = c².*

In [12]:
def euler_9():
    """Special Pythagorean Triplet"""
    for a in range(1, 1000):
        for b in range(a, 1000 - a):
            c = 1000 - a - b
            if c > b and a * a + b * b == c * c:
                return a * b * c


run(euler_9)

  9: Special Pythagorean Triplet                   7 msec ⇒ 31875000          ✅ 

## [Problem 10](https://projecteuler.net/problem=10)

*Sieve all primes below two million and sum them.*

In [13]:
def euler_10():
    """Summation of Primes"""
    return sum(primes_up_to(1_999_999))


run(euler_10)

 10: Summation of Primes                          37 msec ⇒ 142913828922      ✅ 

## [Problem 11](https://projecteuler.net/problem=11)

*Check every 4-in-a-row product horizontally, vertically, and on both diagonals.*

In [14]:
def euler_11():
    """Largest Product in a Grid"""
    grid = read_grid(DATA[11])
    n = 20
    best = 0
    for r in range(n):
        for c in range(n):
            if c <= n - 4:
                best = max(best, prod(grid[r][c + i] for i in range(4)))
            if r <= n - 4:
                best = max(best, prod(grid[r + i][c] for i in range(4)))
            if r <= n - 4 and c <= n - 4:
                best = max(best, prod(grid[r + i][c + i] for i in range(4)))
            if r <= n - 4 and c >= 3:
                best = max(best, prod(grid[r + i][c - i] for i in range(4)))
    return best


run(euler_11)

 11: Largest Product in a Grid                     0 msec ⇒ 70600674          ✅ 

## [Problem 12](https://projecteuler.net/problem=12)

*Generate triangular numbers, counting divisors via factorization until d(t) > 500.*

In [15]:
def euler_12():
    """Highly Divisible Triangular Number"""
    n = t = 0
    while True:
        n += 1
        t += n
        if num_divisors(t) > 500:
            return t


run(euler_12)

 12: Highly Divisible Triangular Number           83 msec ⇒ 76576500          ✅ 

## [Problem 13](https://projecteuler.net/problem=13)

*Sum the hundred 50-digit numbers and take the first ten digits.*

In [16]:
def euler_13():
    """Large Sum"""
    nums = [int(line) for line in DATA[13].split()]
    return int(str(sum(nums))[:10])


run(euler_13)

 13: Large Sum                                     0 msec ⇒ 5537376230        ✅ 

## [Problem 14](https://projecteuler.net/problem=14)

*Walk Collatz chains with a memo cache, tracking the longest start below one million.*

In [17]:
def euler_14():
    """Longest Collatz Sequence"""
    limit = 1_000_000
    cache = {1: 1}
    best_len = best_n = 0
    for start in range(2, limit):
        path = []
        m = start
        while m not in cache:
            path.append(m)
            m = m // 2 if m % 2 == 0 else 3 * m + 1
        base = cache[m]
        for i, v in enumerate(reversed(path)):
            cache[v] = base + i + 1
        if cache[start] > best_len:
            best_len, best_n = cache[start], start
    return best_n


run(euler_14)

 14: Longest Collatz Sequence                    625 msec ⇒ 837799            ✅ 

## [Problem 15](https://projecteuler.net/problem=15)

*The number of lattice paths is the central binomial coefficient C(40, 20).*

In [18]:
def euler_15():
    """Lattice Paths"""
    return comb(40, 20)


run(euler_15)

 15: Lattice Paths                                 0 msec ⇒ 137846528820      ✅ 

## [Problem 16](https://projecteuler.net/problem=16)

*Digit sum of the big integer 2¹⁰⁰⁰.*

In [19]:
def euler_16():
    """Power Digit Sum"""
    return digit_sum(2 ** 1000)


run(euler_16)

 16: Power Digit Sum                               0 msec ⇒ 1366              ✅ 

## [Problem 17](https://projecteuler.net/problem=17)

*Spell each number 1..1000 into words and count the letters (no spaces).*

In [20]:
def euler_17():
    """Number Letter Counts"""
    return sum(len(number_to_words(i).replace(" ", "")) for i in range(1, 1001))


run(euler_17)

 17: Number Letter Counts                          0 msec ⇒ 21124             ✅ 

## [Problem 18](https://projecteuler.net/problem=18)

*Bottom-up DP: collapse the triangle row by row taking the better child.*

In [21]:
def euler_18():
    """Maximum Path Sum I"""
    return _max_triangle_path(read_grid(DATA[18]))


run(euler_18)

 18: Maximum Path Sum I                            0 msec ⇒ 1074              ✅ 

## [Problem 19](https://projecteuler.net/problem=19)

*Iterate months of 1901–2000 with datetime and count those starting on Sunday.*

In [22]:
def euler_19():
    """Counting Sundays"""
    import datetime
    return sum(
        1
        for y in range(1901, 2001)
        for m in range(1, 13)
        if datetime.date(y, m, 1).weekday() == 6
    )


run(euler_19)

 19: Counting Sundays                              0 msec ⇒ 171               ✅ 

## [Problem 20](https://projecteuler.net/problem=20)

*Digit sum of 100! computed with big integers.*

In [23]:
def euler_20():
    """Factorial Digit Sum"""
    return digit_sum(factorial(100))


run(euler_20)

 20: Factorial Digit Sum                           0 msec ⇒ 648               ✅ 

## [Problem 21](https://projecteuler.net/problem=21)

*Sieve proper-divisor sums, then sum members of amicable pairs below 10000.*

In [24]:
def euler_21():
    """Amicable Numbers"""
    s = divisor_sum_sieve(10000)
    total = 0
    for a in range(2, 10000):
        b = s[a]
        if b != a and b < 10000 and s[b] == a:
            total += a
    return total


run(euler_21)

 21: Amicable Numbers                              3 msec ⇒ 31626             ✅ 

## [Problem 22](https://projecteuler.net/problem=22)

*Sort the names, then sum position × alphabetical letter value.*

In [25]:
def euler_22():
    """Names Scores"""
    names = sorted(DATA[22].replace('"', "").split(","))
    return sum(
        i * sum(ord(c) - 64 for c in name)
        for i, name in enumerate(names, 1)
    )


run(euler_22)

 22: Names Scores                                  2 msec ⇒ 871198282         ✅ 

## [Problem 23](https://projecteuler.net/problem=23)

*Sieve abundant numbers, mark every pairwise sum, and add up what can't be formed.*

In [26]:
def euler_23():
    """Non-Abundant Sums"""
    LIMIT = 28123
    s = divisor_sum_sieve(LIMIT)
    abundant = [i for i in range(12, LIMIT + 1) if s[i] > i]
    can = bytearray(LIMIT + 1)
    for i, a in enumerate(abundant):
        for b in abundant[i:]:
            if a + b > LIMIT:
                break
            can[a + b] = 1
    return sum(i for i in range(LIMIT + 1) if not can[i])


run(euler_23)

 23: Non-Abundant Sums                           384 msec ⇒ 4179871           ✅ 

## [Problem 24](https://projecteuler.net/problem=24)

*Take the millionth entry from itertools.permutations of the digits 0–9.*

In [27]:
def euler_24():
    """Lexicographic Permutations"""
    from itertools import islice
    return int("".join(next(islice(permutations("0123456789"), 999999, None))))


run(euler_24)

 24: Lexicographic Permutations                    6 msec ⇒ 2783915460        ✅ 

## [Problem 25](https://projecteuler.net/problem=25)

*Advance the Fibonacci sequence until a term reaches 10⁹⁹⁹; return its index.*

In [28]:
def euler_25():
    """1000-digit Fibonacci Number"""
    for i, f in enumerate(fib(), 1):
        if f >= 10 ** 999:
            return i


run(euler_25)

 25: 1000-digit Fibonacci Number                  15 msec ⇒ 4782              ✅ 

## [Problem 26](https://projecteuler.net/problem=26)

*Simulate long division, detecting the repeating-remainder cycle length for each d < 1000.*

In [29]:
def euler_26():
    """Reciprocal Cycles"""
    def cycle_len(d):
        seen = {}
        r, pos = 1, 0
        while r != 0 and r not in seen:
            seen[r] = pos
            r = (r * 10) % d
            pos += 1
        return 0 if r == 0 else pos - seen[r]
    return max(range(2, 1000), key=cycle_len)


run(euler_26)

 26: Reciprocal Cycles                             7 msec ⇒ 983               ✅ 

## [Problem 27](https://projecteuler.net/problem=27)

*Brute-force coefficients a, b, counting consecutive primes produced by n² + an + b.*

In [30]:
def euler_27():
    """Quadratic Primes"""
    best_len, best_prod = 0, 0
    bs = primes_up_to(1000)
    for a in range(-999, 1000):
        for b in bs:
            n = 0
            while is_prime(n * n + a * n + b):
                n += 1
            if n > best_len:
                best_len, best_prod = n, a * b
    return best_prod


run(euler_27)

 27: Quadratic Primes                          1,300 msec ⇒ -59231            ✅ 🐌

## [Problem 28](https://projecteuler.net/problem=28)

*Sum the ring corners of the number spiral using a closed-form per ring.*

In [31]:
def euler_28():
    """Number Spiral Diagonals"""
    total = 1
    for n in range(3, 1002, 2):
        total += 4 * n * n - 6 * (n - 1)
    return total


run(euler_28)

 28: Number Spiral Diagonals                       0 msec ⇒ 669171001         ✅ 

## [Problem 29](https://projecteuler.net/problem=29)

*Collect a^b into a set for 2 ≤ a, b ≤ 100 and count distinct values.*

In [32]:
def euler_29():
    """Distinct Powers"""
    return len({a ** b for a in range(2, 101) for b in range(2, 101)})


run(euler_29)

 29: Distinct Powers                               2 msec ⇒ 9183              ✅ 

## [Problem 30](https://projecteuler.net/problem=30)

*Search for numbers equal to the sum of the fifth powers of their digits.*

In [33]:
def euler_30():
    """Digit Fifth Powers"""
    p5 = [i ** 5 for i in range(10)]
    return sum(
        n for n in range(10, 354295)
        if n == sum(p5[int(c)] for c in str(n))
    )


run(euler_30)

 30: Digit Fifth Powers                          135 msec ⇒ 443839            ✅ 

## [Problem 31](https://projecteuler.net/problem=31)

*Classic coin-change DP counting the ways to make £2 from the coin set.*

In [34]:
def euler_31():
    """Coin Sums"""
    coins = [1, 2, 5, 10, 20, 50, 100, 200]
    ways = [1] + [0] * 200
    for c in coins:
        for i in range(c, 201):
            ways[i] += ways[i - c]
    return ways[200]


run(euler_31)

 31: Coin Sums                                     0 msec ⇒ 73682             ✅ 

## [Problem 32](https://projecteuler.net/problem=32)

*Search a × b whose concatenation with the product is a 1–9 pandigital; sum distinct products.*

In [35]:
def euler_32():
    """Pandigital Products"""
    products = set()
    for a in range(1, 100):
        for b in range(a, 10000):
            s = str(a) + str(b) + str(a * b)
            if len(s) > 9:
                break
            if len(s) == 9 and set(s) == set("123456789"):
                products.add(a * b)
    return sum(products)


run(euler_32)

 32: Pandigital Products                          15 msec ⇒ 45228             ✅ 

## [Problem 33](https://projecteuler.net/problem=33)

*Find the four nontrivial digit-cancelling fractions; return the reduced product's denominator.*

In [36]:
def euler_33():
    """Digit Cancelling Fractions"""
    p = Fraction(1, 1)
    for d in range(11, 100):
        for n in range(10, d):
            ns, ds = str(n), str(d)
            if ns[1] == ds[0] and ds[1] != "0":
                if Fraction(n, d) == Fraction(int(ns[0]), int(ds[1])):
                    p *= Fraction(n, d)
    return p.denominator


run(euler_33)

 33: Digit Cancelling Fractions                    1 msec ⇒ 100               ✅ 

## [Problem 34](https://projecteuler.net/problem=34)

*Search for numbers equal to the sum of the factorials of their digits.*

In [37]:
def euler_34():
    """Digit Factorials"""
    fact = [factorial(i) for i in range(10)]
    return sum(
        n for n in range(10, 2540161)
        if n == sum(fact[int(c)] for c in str(n))
    )


run(euler_34)

 34: Digit Factorials                          1,040 msec ⇒ 40730             ✅ 🐌

## [Problem 35](https://projecteuler.net/problem=35)

*Sieve to a million, keeping primes all of whose digit rotations are also prime.*

In [38]:
def euler_35():
    """Circular Primes"""
    sieve = prime_sieve(1_000_000)
    count = 0
    for n in range(2, 1_000_000):
        if sieve[n]:
            s = str(n)
            if all(sieve[int(s[i:] + s[:i])] for i in range(len(s))):
                count += 1
    return count


run(euler_35)

 35: Circular Primes                              42 msec ⇒ 55                ✅ 

## [Problem 36](https://projecteuler.net/problem=36)

*Collect numbers palindromic in both base 10 and base 2.*

In [39]:
def euler_36():
    """Double-base Palindromes"""
    return sum(
        n for n in range(1_000_000)
        if is_palindrome(n) and is_palindrome(n, 2)
    )


run(euler_36)

 36: Double-base Palindromes                     280 msec ⇒ 872187            ✅ 

## [Problem 37](https://projecteuler.net/problem=37)

*Search for the eleven primes that stay prime when truncated from either end.*

In [40]:
def euler_37():
    """Truncatable Primes"""
    found = []
    n = 10
    while len(found) < 11:
        n += 1
        if is_prime(n):
            s = str(n)
            if (all(is_prime(int(s[i:])) for i in range(len(s)))
                    and all(is_prime(int(s[:i])) for i in range(1, len(s)))):
                found.append(n)
    return sum(found)


run(euler_37)

 37: Truncatable Primes                          968 msec ⇒ 748317            ✅ 

## [Problem 38](https://projecteuler.net/problem=38)

*Concatenate n, 2n, 3n, … into a 1–9 pandigital and keep the maximum.*

In [41]:
def euler_38():
    """Pandigital Multiples"""
    best = 0
    for n in range(1, 10000):
        s = str(n)
        k = 2
        while len(s) < 9:
            s += str(n * k)
            k += 1
        if len(s) == 9 and set(s) == set("123456789"):
            best = max(best, int(s))
    return best


run(euler_38)

 38: Pandigital Multiples                          3 msec ⇒ 932718654         ✅ 

## [Problem 39](https://projecteuler.net/problem=39)

*Count right triangles by perimeter (p ≤ 1000) and return the most frequent perimeter.*

In [42]:
def euler_39():
    """Integer Right Triangles"""
    counts = Counter()
    for a in range(1, 500):
        for b in range(a, 500):
            c2 = a * a + b * b
            c = isqrt(c2)
            if c * c == c2:
                p = a + b + c
                if p <= 1000:
                    counts[p] += 1
    return counts.most_common(1)[0][0]


run(euler_39)

 39: Integer Right Triangles                       7 msec ⇒ 840               ✅ 

## [Problem 40](https://projecteuler.net/problem=40)

*Build Champernowne's string and multiply the digits at positions 1, 10, …, 10⁶.*

In [43]:
def euler_40():
    """Champernowne's Constant"""
    s = "".join(str(i) for i in range(1, 1_000_000))
    p = 1
    for k in range(7):
        p *= int(s[10 ** k - 1])
    return p


run(euler_40)

 40: Champernowne's Constant                      53 msec ⇒ 210               ✅ 

## [Problem 41](https://projecteuler.net/problem=41)

*Scan pandigital permutations (7-, then 4-digit) top-down for the first prime.*

In [44]:
def euler_41():
    """Pandigital Prime"""
    for n in (7, 4):
        digs = "".join(str(i) for i in range(1, n + 1))
        for p in sorted(permutations(digs), reverse=True):
            num = int("".join(p))
            if is_prime(num):
                return num


run(euler_41)

 41: Pandigital Prime                              0 msec ⇒ 7652413           ✅ 

## [Problem 42](https://projecteuler.net/problem=42)

*Count words whose alphabetical value is a triangular number.*

In [45]:
def euler_42():
    """Coded Triangle Numbers"""
    words = DATA[42].replace('"', "").split(",")
    tri = {triangular(n) for n in range(1, 100)}
    return sum(
        1 for w in words
        if sum(ord(c) - 64 for c in w) in tri
    )


run(euler_42)

 42: Coded Triangle Numbers                        1 msec ⇒ 162               ✅ 

## [Problem 43](https://projecteuler.net/problem=43)

*Test permutations of 0–9 for the substring-divisibility property and sum them.*

In [46]:
def euler_43():
    """Sub-string Divisibility"""
    primes7 = [2, 3, 5, 7, 11, 13, 17]
    total = 0
    for p in permutations("0123456789"):
        if all(int("".join(p[i + 1:i + 4])) % primes7[i] == 0 for i in range(7)):
            total += int("".join(p))
    return total


run(euler_43)

 43: Sub-string Divisibility                   1,098 msec ⇒ 16695334890       ✅ 🐌

## [Problem 44](https://projecteuler.net/problem=44)

*Find the pentagonal pair whose sum and difference are both pentagonal; return the difference.*

In [47]:
def euler_44():
    """Pentagon Numbers"""
    pents = [pentagonal(n) for n in range(1, 3000)]
    pset = set(pents)
    best = None
    for i in range(len(pents)):
        for j in range(i):
            a, b = pents[j], pents[i]
            d = b - a
            if d in pset and (a + b) in pset:
                if best is None or d < best:
                    best = d
    return best


run(euler_44)

 44: Pentagon Numbers                            166 msec ⇒ 5482660           ✅ 

## [Problem 45](https://projecteuler.net/problem=45)

*Iterate hexagonals (each is triangular) and test each for being pentagonal.*

In [48]:
def euler_45():
    """Triangular, Pentagonal, and Hexagonal"""
    n = 144
    while True:
        n += 1
        h = hexagonal(n)
        if is_pentagonal(h):
            return h


run(euler_45)

 45: Triangular, Pentagonal, and Hexagonal         4 msec ⇒ 1533776805        ✅ 

## [Problem 46](https://projecteuler.net/problem=46)

*Find the smallest odd composite not expressible as prime + 2×square.*

In [49]:
def euler_46():
    """Goldbach's Other Conjecture"""
    def can(n):
        for i in range(1, isqrt(n // 2) + 1):
            if is_prime(n - 2 * i * i):
                return True
        return False
    n = 9
    while True:
        n += 2
        if not is_prime(n) and not can(n):
            return n


run(euler_46)

 46: Goldbach's Other Conjecture                  13 msec ⇒ 5777              ✅ 

## [Problem 47](https://projecteuler.net/problem=47)

*Scan for the first run of four consecutive integers each with four distinct prime factors.*

In [50]:
def euler_47():
    """Distinct Primes Factors"""
    n = 1
    consec = 0
    while True:
        n += 1
        if len(factorize(n)) == 4:
            consec += 1
            if consec == 4:
                return n - 3
        else:
            consec = 0


run(euler_47)

 47: Distinct Primes Factors                     253 msec ⇒ 134043            ✅ 

## [Problem 48](https://projecteuler.net/problem=48)

*Sum k^k mod 10¹⁰ using fast modular exponentiation.*

In [51]:
def euler_48():
    """Self Powers"""
    mod = 10 ** 10
    return sum(pow(k, k, mod) for k in range(1, 1001)) % mod


run(euler_48)

 48: Self Powers                                   1 msec ⇒ 9110846700        ✅ 

## [Problem 49](https://projecteuler.net/problem=49)

*Find 4-digit prime arithmetic triples that are digit permutations (skipping the given one).*

In [52]:
def euler_49():
    """Prime Permutations"""
    for a in range(1000, 10000):
        if is_prime(a):
            for d in range(1, 5000):
                b, c = a + d, a + 2 * d
                if c < 10000 and is_prime(b) and is_prime(c) \
                        and is_permutation(a, b) and is_permutation(a, c):
                    if a != 1487:
                        return int(f"{a}{b}{c}")


run(euler_49)

 49: Prime Permutations                          753 msec ⇒ 296962999629      ✅ 

## [Problem 50](https://projecteuler.net/problem=50)

*Use prefix sums of primes to find the longest consecutive run summing to a prime below a million.*

In [53]:
def euler_50():
    """Consecutive Prime Sum"""
    primes = primes_up_to(1_000_000)
    pset = set(primes)
    pre = [0]
    for p in primes:
        pre.append(pre[-1] + p)
    best_len, best_prime = 0, 0
    n = len(primes)
    for i in range(n):
        for j in range(i + best_len, n):
            s = pre[j + 1] - pre[i]
            if s >= 1_000_000:
                break
            if s in pset and (j + 1 - i) > best_len:
                best_len, best_prime = j + 1 - i, s
    return best_prime


run(euler_50)

 50: Consecutive Prime Sum                        35 msec ⇒ 997651            ✅ 

## [Problem 51](https://projecteuler.net/problem=51)

*Group primes by wildcard digit patterns and find a pattern with an eight-prime family.*

In [54]:
def euler_51():
    """Prime Digit Replacements"""
    for length in range(2, 8):
        families = defaultdict(list)
        for p in primes_in_range(10 ** (length - 1), 10 ** length - 1):
            s = str(p)
            for r in range(1, length):
                for combo in combinations(range(length), r):
                    if len({s[i] for i in combo}) == 1:
                        pat = "".join(
                            "*" if i in combo else s[i] for i in range(length)
                        )
                        families[pat].append(p)
        best = None
        for members in families.values():
            if len(members) >= 8:
                cand = min(members)
                if best is None or cand < best:
                    best = cand
        if best is not None:
            return best


run(euler_51)

 51: Prime Digit Replacements                  1,167 msec ⇒ 121313            ✅ 🐌

## [Problem 52](https://projecteuler.net/problem=52)

*Search for the smallest x whose multiples 2x..6x are all digit permutations of x.*

In [55]:
def euler_52():
    """Permuted Multiples"""
    x = 1
    while True:
        x += 1
        if all(is_permutation(x, k * x) for k in range(2, 7)):
            return x


run(euler_52)

 52: Permuted Multiples                           69 msec ⇒ 142857            ✅ 

## [Problem 53](https://projecteuler.net/problem=53)

*Count the binomial coefficients C(n, r) exceeding one million for n ≤ 100.*

In [56]:
def euler_53():
    """Combinatoric Selections"""
    return sum(
        1
        for n in range(1, 101)
        for r in range(n + 1)
        if comb(n, r) > 1_000_000
    )


run(euler_53)

 53: Combinatoric Selections                       0 msec ⇒ 4075              ✅ 

## [Problem 54](https://projecteuler.net/problem=54)

*Score each poker hand by category and tiebreakers; count player one's wins.*

In [57]:
def euler_54():
    """Poker Hands"""
    ORDER = "23456789TJQKA"

    def rank(hand):
        vals = sorted((ORDER.index(c[0]) for c in hand), reverse=True)
        suits = [c[1] for c in hand]
        cnt = Counter(vals)
        order = sorted(vals, key=lambda v: (cnt[v], v), reverse=True)
        counts = sorted(cnt.values(), reverse=True)
        flush = len(set(suits)) == 1
        straight = len(cnt) == 5 and max(vals) - min(vals) == 4
        if straight and flush:
            cat = 8
        elif counts == [4, 1]:
            cat = 7
        elif counts == [3, 2]:
            cat = 6
        elif flush:
            cat = 5
        elif straight:
            cat = 4
        elif counts == [3, 1, 1]:
            cat = 3
        elif counts == [2, 2, 1]:
            cat = 2
        elif counts == [2, 1, 1, 1]:
            cat = 1
        else:
            cat = 0
        return (cat, order)

    count = 0
    for line in DATA[54].splitlines():
        cards = line.split()
        if rank(cards[:5]) > rank(cards[5:]):
            count += 1
    return count


run(euler_54)

 54: Poker Hands                                   4 msec ⇒ 376               ✅ 

## [Problem 55](https://projecteuler.net/problem=55)

*Count Lychrel numbers below 10000 (no palindrome within 50 reverse-and-add steps).*

In [58]:
def euler_55():
    """Lychrel Numbers"""
    def is_lychrel(n):
        for _ in range(50):
            n += int(str(n)[::-1])
            if is_palindrome(n):
                return False
        return True
    return sum(1 for n in range(1, 10000) if is_lychrel(n))


run(euler_55)

 55: Lychrel Numbers                              22 msec ⇒ 249               ✅ 

## [Problem 56](https://projecteuler.net/problem=56)

*Maximize the digit sum of a^b for a, b < 100.*

In [59]:
def euler_56():
    """Powerful Digit Sum"""
    return max(digit_sum(a ** b) for a in range(1, 100) for b in range(1, 100))


run(euler_56)

 56: Powerful Digit Sum                           48 msec ⇒ 972               ✅ 

## [Problem 57](https://projecteuler.net/problem=57)

*Iterate √2 continued-fraction convergents and count those with a longer numerator than denominator.*

In [60]:
def euler_57():
    """Square Root Convergents"""
    n, d = 3, 2
    count = 0
    for _ in range(1000):
        if len(str(n)) > len(str(d)):
            count += 1
        n, d = n + 2 * d, n + d
    return count


run(euler_57)

 57: Square Root Convergents                       1 msec ⇒ 153               ✅ 

## [Problem 58](https://projecteuler.net/problem=58)

*Grow the spiral, tracking the prime ratio along the diagonals until it drops below 10%.*

In [61]:
def euler_58():
    """Spiral Primes"""
    total, primes, side = 1, 0, 1
    while True:
        side += 2
        corners = [side * side - k * (side - 1) for k in range(4)]
        primes += sum(1 for c in corners if is_prime(c))
        total += 4
        if primes * 10 < total:
            return side


run(euler_58)

 58: Spiral Primes                                74 msec ⇒ 26241             ✅ 

## [Problem 59](https://projecteuler.net/problem=59)

*Recover the 3-character XOR key by frequency analysis (space is most common) and sum the plaintext.*

In [62]:
def euler_59():
    """XOR Decryption"""
    data = [int(x) for x in DATA[59].split(",")]
    key = []
    for i in range(3):
        col = data[i::3]
        common = Counter(col).most_common(1)[0][0]
        key.append(common ^ 32)
    return sum(c ^ key[i % 3] for i, c in enumerate(data))


run(euler_59)

 59: XOR Decryption                                0 msec ⇒ 107359            ✅ 

## [Problem 60](https://projecteuler.net/problem=60)

*DFS for five primes whose every pairwise concatenation is prime; minimize the sum.*

In [63]:
def euler_60():
    """Prime Pair Sets"""
    @lru_cache(maxsize=None)
    def good(a, b):
        return is_prime(int(f"{a}{b}")) and is_prime(int(f"{b}{a}"))

    primes = [p for p in primes_up_to(10000) if p not in (2, 5)]
    best = [float("inf")]

    def search(chain, chain_sum, candidates):
        if len(chain) == 5:
            best[0] = min(best[0], chain_sum)
            return
        for i, p in enumerate(candidates):
            if chain_sum + p >= best[0]:
                break
            if all(good(p, c) for c in chain):
                newc = [q for q in candidates[i + 1:] if good(p, q)]
                search(chain + [p], chain_sum + p, newc)

    search([], 0, primes)
    return best[0]


run(euler_60)

 60: Prime Pair Sets                           1,548 msec ⇒ 26033             ✅ 🐌

## [Problem 61](https://projecteuler.net/problem=61)

*DFS chaining 4-digit polygonal numbers (triangle..octagonal) into a cyclic set of six.*

In [64]:
def euler_61():
    """Cyclical Figurate Numbers"""
    def poly(s, n):
        return n * ((s - 2) * n - (s - 4)) // 2

    polys = {}
    for s in range(3, 9):
        nums = []
        n = 1
        while True:
            v = poly(s, n)
            if v >= 10000:
                break
            if v >= 1000:
                nums.append(v)
            n += 1
        polys[s] = nums

    def search(chain, used):
        if len(chain) == 6:
            if str(chain[-1])[2:] == str(chain[0])[:2]:
                return chain
            return None
        last = str(chain[-1])[2:]
        for s in range(3, 9):
            if s in used:
                continue
            for num in polys[s]:
                if str(num)[:2] == last:
                    res = search(chain + [num], used | {s})
                    if res:
                        return res
        return None

    for start in polys[8]:
        res = search([start], {8})
        if res:
            return sum(res)


run(euler_61)

 61: Cyclical Figurate Numbers                     1 msec ⇒ 28684             ✅ 

## [Problem 62](https://projecteuler.net/problem=62)

*Group cubes by their sorted digits and return the smallest of the first five-member group.*

In [65]:
def euler_62():
    """Cubic Permutations"""
    groups = defaultdict(list)
    n = 0
    while True:
        n += 1
        c = n ** 3
        key = "".join(sorted(str(c)))
        groups[key].append(c)
        if len(groups[key]) == 5:
            return min(groups[key])


run(euler_62)

 62: Cubic Permutations                            4 msec ⇒ 127035954683      ✅ 

## [Problem 63](https://projecteuler.net/problem=63)

*Count n-digit numbers that are perfect nth powers.*

In [66]:
def euler_63():
    """Powerful Digit Counts"""
    return sum(
        1
        for n in range(1, 30)
        for b in range(1, 10)
        if len(str(b ** n)) == n
    )


run(euler_63)

 63: Powerful Digit Counts                         0 msec ⇒ 49                ✅ 

## [Problem 64](https://projecteuler.net/problem=64)

*Count square roots (n ≤ 10000) whose continued-fraction period is odd.*

In [67]:
def euler_64():
    """Odd Period Square Roots"""
    count = 0
    for n in range(2, 10001):
        _, period = continued_fraction_sqrt(n)
        if period and len(period) % 2 == 1:
            count += 1
    return count


run(euler_64)

 64: Odd Period Square Roots                      20 msec ⇒ 1322              ✅ 

## [Problem 65](https://projecteuler.net/problem=65)

*Build the 100th convergent of e via its continued fraction; return the numerator's digit sum.*

In [68]:
def euler_65():
    """Convergents of e"""
    terms = [2]
    k = 1
    while len(terms) < 100:
        terms += [1, 2 * k, 1]
        k += 1
    terms = terms[:100]
    val = Fraction(terms[-1])
    for a in reversed(terms[:-1]):
        val = a + 1 / val
    return digit_sum(val.numerator)


run(euler_65)

 65: Convergents of e                              0 msec ⇒ 272               ✅ 

## [Problem 66](https://projecteuler.net/problem=66)

*Solve each Pell equation via continued fractions; return the D with the largest minimal solution.*

In [69]:
def euler_66():
    """Diophantine Equation"""
    best_x, best_D = 0, 0
    for D in range(2, 1001):
        a0, period = continued_fraction_sqrt(D)
        if not period:
            continue
        h_prev, h = 1, a0
        k_prev, k = 0, 1
        i = 0
        while h * h - D * k * k != 1:
            a = period[i % len(period)]
            i += 1
            h_prev, h = h, a * h + h_prev
            k_prev, k = k, a * k + k_prev
        if h > best_x:
            best_x, best_D = h, D
    return best_D


run(euler_66)

 66: Diophantine Equation                          2 msec ⇒ 661               ✅ 

## [Problem 67](https://projecteuler.net/problem=67)

*Same bottom-up triangle DP as Problem 18, on the large bundled triangle.*

In [70]:
def euler_67():
    """Maximum Path Sum II"""
    rows = read_grid(DATA[67])
    return _max_triangle_path(rows)


run(euler_67)

 67: Maximum Path Sum II                           0 msec ⇒ 7273              ✅ 

## [Problem 68](https://projecteuler.net/problem=68)

*Search permutations of 1–10 for the maximum 16-digit magic 5-gon ring string.*

In [71]:
def euler_68():
    """Magic 5-gon Ring"""
    best = 0
    universe = set(range(1, 11))
    for inner in permutations(range(1, 11), 5):
        if 10 in inner:
            continue
        outer_set = universe - set(inner)
        s = sum(inner)
        if (55 + s) % 5 != 0:
            continue
        L = (55 + s) // 5
        outer = []
        ok = True
        for i in range(5):
            o = L - inner[i] - inner[(i + 1) % 5]
            if o not in outer_set or o in outer:
                ok = False
                break
            outer.append(o)
        if not ok or set(outer) != outer_set:
            continue
        start = outer.index(min(outer))
        groups = [
            (outer[(start + i) % 5], inner[(start + i) % 5],
             inner[(start + i + 1) % 5])
            for i in range(5)
        ]
        text = "".join(str(x) for g in groups for x in g)
        if len(text) == 16:
            best = max(best, int(text))
    return best


run(euler_68)

 68: Magic 5-gon Ring                              5 msec ⇒ 6531031914842725  ✅ 

## [Problem 69](https://projecteuler.net/problem=69)

*Maximize n/φ(n) by multiplying successive primes (primorial) while staying under a million.*

In [72]:
def euler_69():
    """Totient Maximum"""
    result, i = 1, 1
    while True:
        p = nth_prime(i)
        if result * p > 1_000_000:
            return result
        result *= p
        i += 1


run(euler_69)

 69: Totient Maximum                               0 msec ⇒ 510510            ✅ 

## [Problem 70](https://projecteuler.net/problem=70)

*Search n = p·q with φ(n) a permutation of n, minimizing the ratio n/φ(n).*

In [73]:
def euler_70():
    """Totient Permutation"""
    LIMIT = 10 ** 7
    primes = primes_in_range(2000, 5000)
    best_ratio, best_n = 10.0, 0
    for i, p in enumerate(primes):
        for q in primes[i:]:
            n = p * q
            if n >= LIMIT:
                break
            phi = (p - 1) * (q - 1)
            if is_permutation(n, phi):
                ratio = n / phi
                if ratio < best_ratio:
                    best_ratio, best_n = ratio, n
    return best_n


run(euler_70)

 70: Totient Permutation                          12 msec ⇒ 8319823           ✅ 

## [Problem 71](https://projecteuler.net/problem=71)

*Scan denominators for the fraction just below 3/7 (mediant / Farey neighbour).*

In [74]:
def euler_71():
    """Ordered Fractions"""
    n, d = 0, 1
    for b in range(1, 1_000_001):
        a = (3 * b - 1) // 7
        if a * d > n * b:
            n, d = a, b
    return n


run(euler_71)

 71: Ordered Fractions                            62 msec ⇒ 428570            ✅ 

## [Problem 72](https://projecteuler.net/problem=72)

*The count of reduced proper fractions is the sum of Euler's totient up to a million.*

In [75]:
def euler_72():
    """Counting Fractions"""
    phi = totient_sieve(1_000_000)
    return sum(phi[2:])


run(euler_72)

 72: Counting Fractions                          382 msec ⇒ 303963552391      ✅ 

## [Problem 73](https://projecteuler.net/problem=73)

*Count reduced fractions strictly between 1/3 and 1/2 for denominators up to 12000.*

In [76]:
def euler_73():
    """Counting Fractions in a Range"""
    count = 0
    for d in range(2, 12001):
        for n in range(d // 3 + 1, (d - 1) // 2 + 1):
            if gcd(n, d) == 1:
                count += 1
    return count


run(euler_73)

 73: Counting Fractions in a Range               730 msec ⇒ 7295372           ✅ 

## [Problem 74](https://projecteuler.net/problem=74)

*Memoize digit-factorial chain lengths and count starts with a chain of exactly 60.*

In [77]:
def euler_74():
    """Digit Factorial Chains"""
    fact = [factorial(i) for i in range(10)]

    def nxt(n):
        return sum(fact[int(c)] for c in str(n))

    cache = {}

    def chain_len(n):
        seen = {}
        cur, steps = n, 0
        while cur not in seen:
            if cur in cache:
                total = steps + cache[cur]
                for k, v in seen.items():
                    cache[k] = total - v
                return total
            seen[cur] = steps
            cur = nxt(cur)
            steps += 1
        total = steps
        for k, v in seen.items():
            cache[k] = total - v
        return total

    return sum(1 for n in range(1, 1_000_000) if chain_len(n) == 60)


run(euler_74)

 74: Digit Factorial Chains                      630 msec ⇒ 402               ✅ 

## [Problem 75](https://projecteuler.net/problem=75)

*Generate primitive Pythagorean triples (Euclid's formula) and count perimeters formed exactly once.*

In [78]:
def euler_75():
    """Singular Integer Right Triangles"""
    LIMIT = 1_500_000
    counts = bytearray(LIMIT + 1)
    m = 2
    while 2 * m * m <= LIMIT:
        for n in range(1, m):
            if (m - n) % 2 == 1 and gcd(m, n) == 1:
                p = 2 * m * (m + n)
                if p <= LIMIT:
                    for k in range(p, LIMIT + 1, p):
                        if counts[k] < 255:
                            counts[k] += 1
        m += 1
    return sum(1 for c in counts if c == 1)


run(euler_75)

 75: Singular Integer Right Triangles            100 msec ⇒ 161667            ✅ 

## [Problem 76](https://projecteuler.net/problem=76)

*Partition-count DP using parts 1..99 (summations of at least two positives).*

In [79]:
def euler_76():
    """Counting Summations"""
    target = 100
    ways = [1] + [0] * target
    for i in range(1, target):
        for j in range(i, target + 1):
            ways[j] += ways[j - i]
    return ways[target]


run(euler_76)

 76: Counting Summations                           0 msec ⇒ 190569291         ✅ 

## [Problem 77](https://projecteuler.net/problem=77)

*Prime-partition DP, increasing n until the number of prime summations exceeds 5000.*

In [80]:
def euler_77():
    """Prime Summations"""
    primes = primes_up_to(100)
    n = 1
    while True:
        n += 1
        ways = [1] + [0] * n
        for p in primes:
            if p > n:
                break
            for j in range(p, n + 1):
                ways[j] += ways[j - p]
        if ways[n] > 5000:
            return n


run(euler_77)

 77: Prime Summations                              1 msec ⇒ 71                ✅ 

## [Problem 78](https://projecteuler.net/problem=78)

*Compute the partition function via the pentagonal-number recurrence mod 10⁶ until a value is 0.*

In [81]:
def euler_78():
    """Coin Partitions"""
    mod = 10 ** 6
    p = [1]
    n = 0
    while True:
        n += 1
        total = 0
        k = 1
        while True:
            g1 = k * (3 * k - 1) // 2
            g2 = k * (3 * k + 1) // 2
            if g1 > n:
                break
            sign = 1 if k % 2 == 1 else -1
            total += sign * p[n - g1]
            if g2 <= n:
                total += sign * p[n - g2]
            k += 1
        total %= mod
        p.append(total)
        if total == 0:
            return n


run(euler_78)

 78: Coin Partitions                           1,209 msec ⇒ 55374             ✅ 🐌

## [Problem 79](https://projecteuler.net/problem=79)

*Topologically sort the digit successor constraints from the keylog into the passcode.*

In [82]:
def euler_79():
    """Passcode Derivation"""
    succ = defaultdict(set)
    chars = set()
    for tok in DATA[79].split():
        a, b, c = tok[0], tok[1], tok[2]
        chars |= {a, b, c}
        succ[a].add(b)
        succ[b].add(c)
    result = ""
    remaining = set(chars)
    while remaining:
        for ch in sorted(remaining):
            if not any(ch in succ[o] for o in remaining):
                result += ch
                remaining.discard(ch)
                break
        else:
            break
    return int(result)


run(euler_79)

 79: Passcode Derivation                           0 msec ⇒ 73162890          ✅ 

## [Problem 80](https://projecteuler.net/problem=80)

*Take integer square roots at high precision and sum the first 100 digits of each non-square.*

In [83]:
def euler_80():
    """Square Root Digital Expansion"""
    total = 0
    for n in range(1, 101):
        r = isqrt(n)
        if r * r == n:
            continue
        val = isqrt(n * 10 ** (2 * 110))
        total += sum(int(c) for c in str(val)[:100])
    return total


run(euler_80)

 80: Square Root Digital Expansion                 0 msec ⇒ 40886             ✅ 

## [Problem 81](https://projecteuler.net/problem=81)

*DP moving only right/down through the matrix, minimizing the path sum.*

In [84]:
def euler_81():
    """Path Sum: Two Ways"""
    g = _read_matrix(DATA[81])
    n = len(g)
    dp = [[0] * n for _ in range(n)]
    for r in range(n - 1, -1, -1):
        for c in range(n - 1, -1, -1):
            if r == n - 1 and c == n - 1:
                dp[r][c] = g[r][c]
            elif r == n - 1:
                dp[r][c] = g[r][c] + dp[r][c + 1]
            elif c == n - 1:
                dp[r][c] = g[r][c] + dp[r + 1][c]
            else:
                dp[r][c] = g[r][c] + min(dp[r + 1][c], dp[r][c + 1])
    return dp[0][0]


run(euler_81)

 81: Path Sum                                      1 msec ⇒ 427337            ✅ 

## [Problem 82](https://projecteuler.net/problem=82)

*Column-by-column DP allowing up, down, and right moves across the matrix.*

In [85]:
def euler_82():
    """Path Sum: Three Ways"""
    g = _read_matrix(DATA[82])
    n = len(g)
    cost = [g[r][n - 1] for r in range(n)]
    for c in range(n - 2, -1, -1):
        new = [0] * n
        new[0] = g[0][c] + cost[0]
        for r in range(1, n):
            new[r] = g[r][c] + min(new[r - 1], cost[r])
        for r in range(n - 2, -1, -1):
            new[r] = min(new[r], g[r][c] + new[r + 1])
        cost = new
    return min(cost)


run(euler_82)

 82: Path Sum                                      1 msec ⇒ 260324            ✅ 

## [Problem 83](https://projecteuler.net/problem=83)

*Dijkstra over the grid with movement in all four directions.*

In [86]:
def euler_83():
    """Path Sum: Four Ways"""
    g = _read_matrix(DATA[83])
    n = len(g)
    INF = float("inf")
    dist = [[INF] * n for _ in range(n)]
    dist[0][0] = g[0][0]
    pq = [(g[0][0], 0, 0)]
    while pq:
        d, r, c = heappop(pq)
        if d > dist[r][c]:
            continue
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n:
                nd = d + g[nr][nc]
                if nd < dist[nr][nc]:
                    dist[nr][nc] = nd
                    heappush(pq, (nd, nr, nc))
    return dist[n - 1][n - 1]


run(euler_83)

 83: Path Sum                                      4 msec ⇒ 425185            ✅ 

## [Problem 84](https://projecteuler.net/problem=84)

*Build the Monopoly Markov transition matrix (cards, jail, doubles) and find the stationary distribution.*

In [87]:
def euler_84():
    """Monopoly Odds"""
    railways = [5, 15, 25, 35]
    utilities = [12, 28]

    def next_in(pos, lst):
        for x in lst:
            if x > pos:
                return x
        return lst[0]

    def resolve(land):
        if land == 30:
            return {10: 1.0}
        if land in (7, 22, 36):
            res = defaultdict(float)
            per = 1 / 16
            for d in (0, 10, 11, 24, 39, 5):
                res[d] += per
            res[next_in(land, railways)] += 2 * per
            res[next_in(land, utilities)] += per
            for d, pr in resolve((land - 3) % 40).items():
                res[d] += per * pr
            res[land] += 6 * per
            return res
        if land in (2, 17, 33):
            res = defaultdict(float)
            per = 1 / 16
            res[0] += per
            res[10] += per
            res[land] += 14 * per
            return res
        return {land: 1.0}

    dist = defaultdict(int)
    for a in range(1, 5):
        for b in range(1, 5):
            dist[a + b] += 1
    T = np.zeros((40, 40))
    for s in range(40):
        for roll, cnt in dist.items():
            p = cnt / 16
            for dest, pp in resolve((s + roll) % 40).items():
                T[s][dest] += p * pp
    v = np.ones(40) / 40
    for _ in range(3000):
        v = v @ T
    top = sorted(range(40), key=lambda i: v[i], reverse=True)[:3]
    return int("".join(f"{i:02d}" for i in top))


run(euler_84)

 84: Monopoly Odds                                 2 msec ⇒ 101524            ✅ 

## [Problem 85](https://projecteuler.net/problem=85)

*Closed-form rectangle count over grid sizes, minimizing the distance to two million.*

In [88]:
def euler_85():
    """Counting Rectangles"""
    target = 2_000_000
    best = None
    for m in range(1, 100):
        for n in range(1, 100):
            cnt = (m * (m + 1) // 2) * (n * (n + 1) // 2)
            diff = abs(cnt - target)
            if best is None or diff < best[0]:
                best = (diff, m * n)
    return best[1]


run(euler_85)

 85: Counting Rectangles                           1 msec ⇒ 2772              ✅ 

## [Problem 86](https://projecteuler.net/problem=86)

*Count cuboids with an integer shortest surface path, growing M until the running total passes a million.*

In [89]:
def euler_86():
    """Cuboid Route"""
    LIMIT = 1_000_000
    M, total = 0, 0
    while total <= LIMIT:
        M += 1
        c = M
        for ab in range(2, 2 * c + 1):
            d2 = ab * ab + c * c
            d = isqrt(d2)
            if d * d == d2:
                if ab <= c:
                    total += ab // 2
                else:
                    total += ab // 2 - (ab - c - 1)
    return M


run(euler_86)

 86: Cuboid Route                                190 msec ⇒ 1818              ✅ 

## [Problem 87](https://projecteuler.net/problem=87)

*Sum the distinct values a² + b³ + c⁴ below fifty million over prime a, b, c.*

In [90]:
def euler_87():
    """Prime Power Triples"""
    LIMIT = 50_000_000
    primes = primes_up_to(isqrt(LIMIT) + 1)
    found = set()
    for c in primes:
        c4 = c ** 4
        if c4 >= LIMIT:
            break
        for b in primes:
            b3 = b ** 3
            if c4 + b3 >= LIMIT:
                break
            for a in primes:
                v = c4 + b3 + a * a
                if v >= LIMIT:
                    break
                found.add(v)
    return len(found)


run(euler_87)

 87: Prime Power Triples                         159 msec ⇒ 1097343           ✅ 

## [Problem 88](https://projecteuler.net/problem=88)

*DFS over factorizations to find the minimal product-sum number for each k, then sum the distinct ones.*

In [91]:
def euler_88():
    """Product-sum Numbers"""
    LIMIT = 12000
    min_ps = [2 * LIMIT] * (LIMIT + 1)

    def search(prod_, summ, cnt, start):
        if cnt >= 2:
            k = prod_ - summ + cnt
            if k <= LIMIT and prod_ < min_ps[k]:
                min_ps[k] = prod_
        for f in range(start, (2 * LIMIT) // prod_ + 1):
            search(prod_ * f, summ + f, cnt + 1, f)

    search(1, 0, 0, 2)
    return sum(set(min_ps[2:]))


run(euler_88)

 88: Product-sum Numbers                          58 msec ⇒ 7587457           ✅ 

## [Problem 89](https://projecteuler.net/problem=89)

*Parse each Roman numeral and re-encode it minimally, summing the characters saved.*

In [92]:
def euler_89():
    """Roman Numerals"""
    vals = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}

    def to_int(s):
        total = 0
        for i, ch in enumerate(s):
            if i + 1 < len(s) and vals[ch] < vals[s[i + 1]]:
                total -= vals[ch]
            else:
                total += vals[ch]
        return total

    table = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"),
             (90, "XC"), (50, "L"), (40, "XL"), (10, "X"), (9, "IX"),
             (5, "V"), (4, "IV"), (1, "I")]

    def to_roman(n):
        res = ""
        for v, sym in table:
            while n >= v:
                res += sym
                n -= v
        return res

    return sum(
        len(line) - len(to_roman(to_int(line)))
        for line in DATA[89].split()
    )


run(euler_89)

 89: Roman Numerals                                1 msec ⇒ 743               ✅ 

## [Problem 90](https://projecteuler.net/problem=90)

*Search pairs of dice faces that can display every two-digit square (with the 6/9 swap).*

In [93]:
def euler_90():
    """Cube Digit Pairs"""
    squares = [(0, 1), (0, 4), (0, 9), (1, 6), (2, 5), (3, 6), (4, 9),
               (6, 4), (8, 1)]

    def has(cube, d):
        if d in (6, 9):
            return 6 in cube or 9 in cube
        return d in cube

    def works(c1, c2):
        return all(
            (has(c1, a) and has(c2, b)) or (has(c1, b) and has(c2, a))
            for a, b in squares
        )

    cubes = list(combinations(range(10), 6))
    count = 0
    for i in range(len(cubes)):
        for j in range(i, len(cubes)):
            if works(cubes[i], cubes[j]):
                count += 1
    return count


run(euler_90)

 90: Cube Digit Pairs                             12 msec ⇒ 1217              ✅ 

## [Problem 91](https://projecteuler.net/problem=91)

*Count right triangles with one vertex at the origin and the others on the grid.*

In [94]:
def euler_91():
    """Right Triangles with Integer Coordinates"""
    N = 50
    pts = [(x, y) for x in range(N + 1) for y in range(N + 1)
           if (x, y) != (0, 0)]

    def right(p, q):
        o = (0, 0)
        for a, b, c in ((o, p, q), (p, o, q), (q, o, p)):
            v1 = (b[0] - a[0], b[1] - a[1])
            v2 = (c[0] - a[0], c[1] - a[1])
            if v1[0] * v2[0] + v1[1] * v2[1] == 0:
                return True
        return False

    count = 0
    for i in range(len(pts)):
        for j in range(i + 1, len(pts)):
            if right(pts[i], pts[j]):
                count += 1
    return count


run(euler_91)

 91: Right Triangles with Integer Coordinates  1,273 msec ⇒ 14234             ✅ 🐌

## [Problem 92](https://projecteuler.net/problem=92)

*Precompute square-digit-chain outcomes for 1..567 and count starts reaching 89.*

In [95]:
def euler_92():
    """Square Digit Chains"""
    def sds(n):
        return sum(int(c) ** 2 for c in str(n))
    outcome = [0] * 568
    for n in range(1, 568):
        m = n
        while m != 1 and m != 89:
            m = sds(m)
        outcome[n] = m
    return sum(1 for n in range(1, 10_000_000) if outcome[sds(n)] == 89)


run(euler_92)

 92: Square Digit Chains                       4,565 msec ⇒ 8581146           ✅ 🐌

## [Problem 93](https://projecteuler.net/problem=93)

*Try all operator and parenthesization combinations on each digit set for the longest run 1..n.*

In [96]:
def euler_93():
    """Arithmetic Expressions"""
    OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul}

    def ap(o, x, y):
        if x is None or y is None:
            return None
        if o == "/":
            return x / y if y != 0 else None
        return OPS[o](x, y)

    def values(nums):
        vals = set()
        for p in permutations(nums):
            a, b, c, d = map(Fraction, p)
            for o1, o2, o3 in product("+-*/", repeat=3):
                for v in (
                    ap(o3, ap(o2, ap(o1, a, b), c), d),
                    ap(o3, ap(o1, a, b), ap(o2, c, d)),
                    ap(o1, a, ap(o3, ap(o2, b, c), d)),
                    ap(o1, a, ap(o2, b, ap(o3, c, d))),
                    ap(o2, ap(o1, a, b), ap(o3, c, d)),
                ):
                    if v is not None and v.denominator == 1 and v > 0:
                        vals.add(int(v))
        return vals

    best_len, best_digits = 0, ""
    for combo in combinations(range(1, 10), 4):
        vals = values(combo)
        n = 1
        while n in vals:
            n += 1
        if n - 1 > best_len:
            best_len = n - 1
            best_digits = "".join(map(str, combo))
    return int(best_digits)


run(euler_93)

 93: Arithmetic Expressions                    1,035 msec ⇒ 1258              ✅ 🐌

## [Problem 94](https://projecteuler.net/problem=94)

*Use two Pell-like recurrences to sum perimeters of near-equilateral integer-area triangles.*

In [97]:
def euler_94():
    """Almost Equilateral Triangles"""
    LIMIT = 10 ** 9
    total = 0
    prev, cur = 1, 5          # family: base = side + 1
    while 3 * cur + 1 <= LIMIT:
        total += 3 * cur + 1
        prev, cur = cur, 14 * cur - prev - 4
    prev, cur = 1, 17         # family: base = side - 1
    while 3 * cur - 1 <= LIMIT:
        total += 3 * cur - 1
        prev, cur = cur, 14 * cur - prev + 4
    return total


run(euler_94)

 94: Almost Equilateral Triangles                  0 msec ⇒ 518408346         ✅ 

## [Problem 95](https://projecteuler.net/problem=95)

*Follow proper-divisor-sum chains to find the longest amicable cycle; return its smallest member.*

In [98]:
def euler_95():
    """Amicable Chains"""
    LIMIT = 10 ** 6
    s = divisor_sum_sieve(LIMIT)
    best_len, best_min = 0, 0
    for start in range(2, LIMIT + 1):
        seen = {}
        n, step = start, 0
        while n <= LIMIT and n > 1 and n not in seen:
            seen[n] = step
            n = s[n]
            step += 1
        if n in seen:
            cycle = [k for k, v in seen.items() if v >= seen[n]]
            if len(cycle) > best_len:
                best_len, best_min = len(cycle), min(cycle)
    return best_min


run(euler_95)

 95: Amicable Chains                           1,813 msec ⇒ 14316             ✅ 🐌

## [Problem 96](https://projecteuler.net/problem=96)

*Bitmask backtracking with a minimum-remaining-values heuristic; sum each grid's top-left three digits.*

In [99]:
def euler_96():
    """Su Doku"""
    def solve(board):
        rows = [0] * 9
        cols = [0] * 9
        boxes = [0] * 9
        for i in range(81):
            v = board[i]
            if v:
                r, c = divmod(i, 9)
                b = (r // 3) * 3 + c // 3
                bit = 1 << v
                rows[r] |= bit
                cols[c] |= bit
                boxes[b] |= bit

        def bt():
            best = -1
            best_mask = 0
            best_count = 10
            for i in range(81):
                if board[i] == 0:
                    r, c = divmod(i, 9)
                    b = (r // 3) * 3 + c // 3
                    cand = (~(rows[r] | cols[c] | boxes[b])) & 0x3FE
                    cnt = bin(cand).count("1")
                    if cnt < best_count:
                        best_count, best, best_mask = cnt, i, cand
                        if cnt == 0:
                            break
            if best == -1:
                return True
            if best_count == 0:
                return False
            i = best
            r, c = divmod(i, 9)
            b = (r // 3) * 3 + c // 3
            mask = best_mask
            while mask:
                bit = mask & (-mask)
                mask -= bit
                v = bit.bit_length() - 1
                board[i] = v
                rows[r] |= bit
                cols[c] |= bit
                boxes[b] |= bit
                if bt():
                    return True
                board[i] = 0
                rows[r] &= ~bit
                cols[c] &= ~bit
                boxes[b] &= ~bit
            return False

        bt()
        return board

    lines = [l.strip() for l in DATA[96].splitlines()
             if l.strip()]
    total = 0
    i = 0
    while i < len(lines):
        if lines[i].startswith("Grid"):
            i += 1
        board = [int(ch) for row in lines[i:i + 9] for ch in row]
        i += 9
        solve(board)
        total += board[0] * 100 + board[1] * 10 + board[2]
    return total


run(euler_96)

 96: Su Doku                                      57 msec ⇒ 24702             ✅ 

## [Problem 97](https://projecteuler.net/problem=97)

*Modular exponentiation to get the last ten digits of 28433·2⁷⁸³⁰⁴⁵⁷ + 1.*

In [100]:
def euler_97():
    """Large Non-Mersenne Prime"""
    mod = 10 ** 10
    return (28433 * pow(2, 7830457, mod) + 1) % mod


run(euler_97)

 97: Large Non-Mersenne Prime                      0 msec ⇒ 8739992577        ✅ 

## [Problem 98](https://projecteuler.net/problem=98)

*Match anagram word pairs against square numbers via digit-to-letter mappings.*

In [101]:
def euler_98():
    """Anagramic Squares"""
    words = DATA[98].replace('"', "").split(",")
    groups = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)
    anagram_sets = [g for g in groups.values() if len(g) >= 2]
    maxlen = max(len(g[0]) for g in anagram_sets)
    sq_by_len = defaultdict(list)
    n = 1
    while len(str(n * n)) <= maxlen:
        sq_by_len[len(str(n * n))].append(n * n)
        n += 1
    best = 0
    for group in anagram_sets:
        L = len(group[0])
        for w1, w2 in permutations(group, 2):
            for sq in sq_by_len[L]:
                s = str(sq)
                m = {}
                ok = True
                for ch, dg in zip(w1, s):
                    if ch in m:
                        if m[ch] != dg:
                            ok = False
                            break
                    else:
                        m[ch] = dg
                if not ok or len(set(m.values())) != len(m):
                    continue
                t = "".join(m[ch] for ch in w2)
                if t[0] == "0":
                    continue
                ti = int(t)
                r = isqrt(ti)
                if r * r == ti:
                    best = max(best, sq, ti)
    return best


run(euler_98)

 98: Anagramic Squares                            49 msec ⇒ 18769             ✅ 

## [Problem 99](https://projecteuler.net/problem=99)

*Compare e·log(b) across the base/exponent pairs to find the largest value.*

In [102]:
def euler_99():
    """Largest Exponential"""
    import math
    best, best_line = 0.0, 0
    for i, line in enumerate(DATA[99].split(), 1):
        b, e = map(int, line.split(","))
        val = e * math.log(b)
        if val > best:
            best, best_line = val, i
    return best_line


run(euler_99)

 99: Largest Exponential                           0 msec ⇒ 709               ✅ 

## [Problem 100](https://projecteuler.net/problem=100)

*Pell-like recurrence generating the blue-disc arrangements past 10¹² discs.*

In [103]:
def euler_100():
    """Arranged Probability"""
    LIMIT = 10 ** 12
    b, t = 15, 21
    while t <= LIMIT:
        b, t = 3 * b + 2 * t - 2, 4 * b + 3 * t - 3
    return b


run(euler_100)

100: Arranged Probability                          0 msec ⇒ 756872327473      ✅ 

# Summary of Runs

In [104]:
runs()

Problems: 100
Run time in seconds: total: 22.7, max: 4.6, mean: 0.227, median: 0.005

  1: Multiples of 3 or 5                           0 msec ⇒ 233168            ✅ 
  2: Even Fibonacci Numbers                        0 msec ⇒ 4613732           ✅ 
  3: Largest Prime Factor                          0 msec ⇒ 6857              ✅ 
  4: Largest Palindrome Product                   25 msec ⇒ 906609            ✅ 
  5: Smallest Multiple                             0 msec ⇒ 232792560         ✅ 
  6: Sum Square Difference                         0 msec ⇒ 25164150          ✅ 
  7: 10001st Prime                                 6 msec ⇒ 104743            ✅ 
  8: Largest Product in a Series                   0 msec ⇒ 23514624000       ✅ 
  9: Special Pythagorean Triplet                   7 msec ⇒ 31875000          ✅ 
 10: Summation of Primes                          37 msec ⇒ 142913828922      ✅ 
 11: Largest Product in a Grid                     0 msec ⇒ 70600674          ✅ 
 12: Highly Divisible T